# VeReMi Paper 1 — Final Model-Safe Feature Schema and Leakage Audit

This notebook freezes the **model-input specification** before any PyTorch Geometric conversion or model training.

### Inputs
Frozen graph dataset:
`GRAPH_DATASET_ROOT / "archives"`

Frozen optimized split:
`BALANCED_SPLIT_ROOT / "06_optimized_full_split_manifest.csv"`

### Output
`FEATURE_SCHEMA_ROOT`

### Main principle
A field being present in the dataset does **not** mean it is automatically a valid model input.

Every column is classified as one of:

- `PRIMARY_MODEL_INPUT`
- `ABLATION_ONLY`
- `TARGET`
- `IDENTIFIER_KEY`
- `TEMPORAL_METADATA`
- `PROVENANCE_METADATA`
- `GROUND_TRUTH_VALIDATION_ONLY`
- `LABEL_DERIVED_LEAKAGE`
- `POTENTIAL_LEAKAGE_REVIEW`
- `GRAPH_CONSTRUCTION_ONLY`
- `UNCLASSIFIED_REVIEW`

The notebook uses a **conservative primary feature set**. Current-snapshot structural statistics are kept for ablation/characterization rather than automatically fed to the primary edge classifier.

### v2 correction
`receiver_speed_magnitude` is explicitly classified as `PRIMARY_MODEL_INPUT`, consistent with the other receiver mobility features.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

from src.config import GRAPH_DATASET_ROOT, BALANCED_SPLIT_ROOT, FEATURE_SCHEMA_ROOT
DATA_ROOT = GRAPH_DATASET_ROOT / "archives"
SPLIT_MANIFEST = BALANCED_SPLIT_ROOT / "06_optimized_full_split_manifest.csv"
OUT = FEATURE_SCHEMA_ROOT
OUT.mkdir(parents=True, exist_ok=True)

archive_dirs = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir()
)

manifest = pd.read_csv(SPLIT_MANIFEST)

assert len(archive_dirs) == 225
assert len(manifest) == 225
assert manifest["archive_name"].nunique() == 225

print("Archives:", len(archive_dirs))
print("Split counts:")
display(manifest["split"].value_counts().to_frame("archives"))
print("Output:", OUT)


## 1. Inventory the exact V1.0 schemas

The audit uses the actual frozen dataset columns. No column names are assumed to exist without checking.


In [ ]:
sample_dir = archive_dirs[0]

node_sample = pd.read_parquet(
    sample_dir / "node_features.parquet"
)

edge_sample = pd.read_parquet(
    sample_dir / "edge_features.parquet"
)

snapshot_sample = pd.read_parquet(
    sample_dir / "snapshot_features.parquet"
)

schemas = []

for table_name, df in [
    ("node", node_sample),
    ("edge", edge_sample),
    ("snapshot", snapshot_sample)
]:
    for col in df.columns:
        schemas.append({
            "table": table_name,
            "column": col,
            "dtype": str(df[col].dtype)
        })

schema_df = pd.DataFrame(schemas)

schema_df.to_csv(
    OUT / "01_exact_schema_inventory.csv",
    index=False
)

display(schema_df)


## 2. Conservative feature-role rules

### Primary node inputs
The main GNN node feature matrix should contain **physical/mobility state, state-change attributes, measurement/noise attributes, and state-availability indicators**.

### Structural topology
Current-snapshot `degree`, `PageRank`, clustering, etc. are useful engineered graph descriptors, but the GNN already receives the adjacency structure. Feeding them to the primary GNN could blur the distinction between **learning from topology** and **manually injecting topology statistics**.

Therefore:
- current topology descriptors → `ABLATION_ONLY`
- lagged topology descriptors (`prev_*`) → `PRIMARY_MODEL_INPUT`

### Graph-level context
For a strict causal prediction protocol:
- current-snapshot graph statistics → `ABLATION_ONLY`
- previous-snapshot graph statistics → `PRIMARY_MODEL_INPUT`

### Ground truth
Any `groundtruth_*` field is used only to verify reconstruction and is **never** a predictor.

### `A_flag`
Because its exact raw-file semantics may be linked to A0/A1 logging provenance, it is conservatively excluded as `POTENTIAL_LEAKAGE_REVIEW` until independently justified.


In [ ]:
# ============================================================
# 2. ROLE CLASSIFICATION
# ============================================================

TARGET_COLUMNS = {
    "attackerType",
    "attacker_type",
    "attack_type",
}

IDENTIFIER_COLUMNS = {
    "archive_name",
    "vehicle_id",
    "sender",
    "receiver",
    "messageID",
    "message_id",
    "snapshot_id",
}

TEMPORAL_METADATA_COLUMNS = {
    "time",
    "sendTime",
    "rcvTime",
    "snapshot_start",
    "snapshot_end",
}

PROVENANCE_PATTERNS = [
    "file",
    "path",
    "folder",
    "archive",
    "log_file",
]

GROUND_TRUTH_PATTERNS = [
    "groundtruth",
    "ground_truth",
]

LABEL_DERIVED_COLUMNS = {
    "labelled_attack_edge_fraction",
    "labeled_attack_edge_fraction",
}

POTENTIAL_LEAKAGE_COLUMNS = {
    "A_flag",
}

CURRENT_NODE_TOPOLOGY = {
    "in_degree",
    "out_degree",
    "degree",
    "degree_centrality",
    "pagerank",
    "clustering_coefficient",
    "component_size",
}

CURRENT_SNAPSHOT_TOPOLOGY = {
    "node_count",
    "isolated_node_count",
    "message_edge_count",
    "unique_directed_edge_count",
    "density",
    "reciprocity",
    "average_degree",
    "average_clustering",
    "connected_components",
    "largest_component_size",
    "largest_component_fraction",
    "mean_RSSI",
    "median_communication_delay",
}

# Fields that are legitimate edge observations/history if present.
EDGE_PRIMARY_CANDIDATES = {
    "RSSI",
    "communication_delay",
    "receiver_state_age",
    "receiver_speed_magnitude",
    "reported_pos_x",
    "reported_pos_y",
    "reported_pos_z",
    "reported_spd_x",
    "reported_spd_y",
    "reported_spd_z",
    "reported_pos_noise_x",
    "reported_pos_noise_y",
    "reported_pos_noise_z",
    "reported_spd_noise_x",
    "reported_spd_noise_y",
    "reported_spd_noise_z",
    "receiver_pos_x",
    "receiver_pos_y",
    "receiver_pos_z",
    "receiver_spd_x",
    "receiver_spd_y",
    "receiver_spd_z",
    "reported_sender_speed_magnitude",
    "reported_distance_to_receiver",
    "relative_velocity_magnitude",
    "prior_pair_message_count",
    "time_since_previous_pair_message",
    "prior_sender_message_count",
    "prior_receiver_message_count",
}

# Node state fields that are appropriate as primary inputs when present.
NODE_PRIMARY_CANDIDATES = {
    "pos_x",
    "pos_y",
    "pos_z",
    "spd_x",
    "spd_y",
    "spd_z",
    "local_noise_x",
    "local_noise_y",
    "local_noise_z",
    "spd_noise_x",
    "spd_noise_y",
    "spd_noise_z",
    "speed_magnitude",
    "motion_angle",
    "delta_speed_magnitude",
    "snapshot_displacement",
    "delta_motion_angle",
    "state_available_current",
    "state_age_snapshots",
}


def has_provenance_pattern(col):
    lc = col.lower()
    return any(p in lc for p in PROVENANCE_PATTERNS)


def has_groundtruth_pattern(col):
    lc = col.lower()
    return any(p in lc for p in GROUND_TRUTH_PATTERNS)


def classify_role(table, col):
    lc = col.lower()

    if col in TARGET_COLUMNS or lc in {x.lower() for x in TARGET_COLUMNS}:
        return "TARGET"

    if col in LABEL_DERIVED_COLUMNS:
        return "LABEL_DERIVED_LEAKAGE"

    if has_groundtruth_pattern(col):
        return "GROUND_TRUTH_VALIDATION_ONLY"

    if col in POTENTIAL_LEAKAGE_COLUMNS:
        return "POTENTIAL_LEAKAGE_REVIEW"

    if col in IDENTIFIER_COLUMNS:
        return "IDENTIFIER_KEY"

    if col in TEMPORAL_METADATA_COLUMNS:
        return "TEMPORAL_METADATA"

    if has_provenance_pattern(col):
        return "PROVENANCE_METADATA"

    if table == "node":
        if col in CURRENT_NODE_TOPOLOGY:
            return "ABLATION_ONLY"

        if col.startswith("prev_") and col.replace("prev_", "", 1) in CURRENT_NODE_TOPOLOGY:
            return "PRIMARY_MODEL_INPUT"

        if col in NODE_PRIMARY_CANDIDATES:
            return "PRIMARY_MODEL_INPUT"

    if table == "edge":
        if col in EDGE_PRIMARY_CANDIDATES:
            return "PRIMARY_MODEL_INPUT"

        # Some raw edge/message fields are necessary to reconstruct the graph
        # but are not predictors.
        if lc in {"type", "a_flag"}:
            return "GRAPH_CONSTRUCTION_ONLY"

    if table == "snapshot":
        if col in CURRENT_SNAPSHOT_TOPOLOGY:
            return "ABLATION_ONLY"

        if col.startswith("prev_"):
            base = col.replace("prev_", "", 1)
            if base in CURRENT_SNAPSHOT_TOPOLOGY:
                return "PRIMARY_MODEL_INPUT"

    return "UNCLASSIFIED_REVIEW"


roles = schema_df.copy()

roles["role"] = roles.apply(
    lambda r: classify_role(
        r["table"],
        r["column"]
    ),
    axis=1
)

roles.to_csv(
    OUT / "02_feature_role_classification.csv",
    index=False
)

display(roles)


## 3. Inspect anything not cleanly classified

The experiment must not proceed while an unexplained field is silently treated as a predictor.


In [ ]:
review_roles = {
    "UNCLASSIFIED_REVIEW",
    "POTENTIAL_LEAKAGE_REVIEW"
}

needs_review = roles[
    roles["role"].isin(
        review_roles
    )
].copy()

needs_review.to_csv(
    OUT / "03_features_requiring_manual_review.csv",
    index=False
)

print(
    "Features requiring review:",
    len(needs_review)
)

display(needs_review)


## 4. Data-type suitability audit

Primary model inputs should be numeric. Any string/object column entering a primary feature list is an automatic failure.


In [ ]:
primary = roles[
    roles["role"] == "PRIMARY_MODEL_INPUT"
].copy()

primary["numeric_dtype"] = ~primary["dtype"].str.lower().str.contains(
    "object|string|str|category"
)

non_numeric_primary = primary[
    ~primary["numeric_dtype"]
].copy()

non_numeric_primary.to_csv(
    OUT / "04_non_numeric_primary_features.csv",
    index=False
)

print(
    "Primary model-input fields:",
    len(primary)
)

print(
    "Non-numeric primary fields:",
    len(non_numeric_primary)
)

display(non_numeric_primary)


## 5. Training-only statistics for primary features

This does **not** fit a scaler yet. It simply verifies that all primary features are present and numerically usable in the **training archives only**.

Validation/test data are not used to estimate preprocessing statistics.


In [ ]:
train_archives = set(
    manifest.loc[
        manifest["split"] == "train",
        "archive_name"
    ]
)

primary_by_table = {
    table: roles[
        (roles["table"] == table)
        &
        (roles["role"] == "PRIMARY_MODEL_INPUT")
    ]["column"].tolist()
    for table in [
        "node",
        "edge",
        "snapshot"
    ]
}

print("Primary node features:", len(primary_by_table["node"]))
print("Primary edge features:", len(primary_by_table["edge"]))
print("Primary snapshot context features:", len(primary_by_table["snapshot"]))


In [ ]:
# Training-only numeric integrity:
# count rows/nulls/infinities for each primary feature.

integrity_records = []

for i, d in enumerate(archive_dirs, 1):

    if d.name not in train_archives:
        continue

    for table, filename in [
        ("node", "node_features.parquet"),
        ("edge", "edge_features.parquet"),
        ("snapshot", "snapshot_features.parquet")
    ]:

        cols = primary_by_table[
            table
        ]

        if not cols:
            continue

        df = pd.read_parquet(
            d / filename,
            columns=cols
        )

        for col in cols:

            x = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            arr = x.to_numpy(
                dtype=float
            )

            integrity_records.append({
                "archive_name":
                    d.name,

                "table":
                    table,

                "column":
                    col,

                "rows":
                    len(x),

                "null_count":
                    int(
                        x.isna().sum()
                    ),

                "positive_inf":
                    int(
                        np.isposinf(
                            arr
                        ).sum()
                    ),

                "negative_inf":
                    int(
                        np.isneginf(
                            arr
                        ).sum()
                    )
            })


integrity = pd.DataFrame(
    integrity_records
)

training_integrity = (
    integrity
    .groupby(
        [
            "table",
            "column"
        ],
        as_index=False
    )
    .agg(
        rows=(
            "rows",
            "sum"
        ),
        null_count=(
            "null_count",
            "sum"
        ),
        positive_inf=(
            "positive_inf",
            "sum"
        ),
        negative_inf=(
            "negative_inf",
            "sum"
        )
    )
)

training_integrity.to_csv(
    OUT /
    "05_training_primary_feature_integrity.csv",
    index=False
)

display(training_integrity)


### Missing-value rule

Expected first-observation history nulls are allowed only for explicitly temporal features such as:

- `delta_*`
- `snapshot_displacement`
- `time_since_previous_pair_message`
- `prev_*`

They are **not** treated as corruption.

For model construction, they will receive a deterministic causal initialization strategy fitted/defined without using validation or test data. This notebook does not apply that transformation yet.


In [ ]:
# ============================================================
# 6. EXPLICITLY FLAG EXPECTED VS UNEXPECTED PRIMARY NULLS
# ============================================================

EXPECTED_TEMPORAL_NULL_FEATURES = {
    "delta_speed_magnitude",
    "delta_motion_angle",
    "snapshot_displacement",
    "time_since_previous_pair_message",
}

def null_type(col):
    if (
        col in EXPECTED_TEMPORAL_NULL_FEATURES
        or
        col.startswith("prev_")
    ):
        return "EXPECTED_CAUSAL_HISTORY_NULL"

    return "UNEXPECTED_NULL_IF_PRESENT"


training_integrity[
    "null_classification"
] = training_integrity[
    "column"
].map(
    null_type
)

training_integrity[
    "integrity_status"
] = np.where(
    (
        training_integrity[
            "positive_inf"
        ] > 0
    )
    |
    (
        training_integrity[
            "negative_inf"
        ] > 0
    ),
    "FAIL_INFINITY",
    np.where(
        (
            training_integrity[
                "null_count"
            ] > 0
        )
        &
        (
            training_integrity[
                "null_classification"
            ]
            ==
            "UNEXPECTED_NULL_IF_PRESENT"
        ),
        "FAIL_UNEXPECTED_NULL",
        "PASS"
    )
)

training_integrity.to_csv(
    OUT /
    "06_training_primary_feature_integrity_classified.csv",
    index=False
)

display(training_integrity)


## 7. Freeze explicit model schemas

The following JSON files become the model-input contract for downstream tensor construction.


In [ ]:
# Conservative primary schemas.

node_primary = roles[
    (roles["table"] == "node")
    &
    (roles["role"] == "PRIMARY_MODEL_INPUT")
]["column"].tolist()

node_ablation = roles[
    (roles["table"] == "node")
    &
    (roles["role"] == "ABLATION_ONLY")
]["column"].tolist()

edge_primary = roles[
    (roles["table"] == "edge")
    &
    (roles["role"] == "PRIMARY_MODEL_INPUT")
]["column"].tolist()

snapshot_primary = roles[
    (roles["table"] == "snapshot")
    &
    (roles["role"] == "PRIMARY_MODEL_INPUT")
]["column"].tolist()

snapshot_ablation = roles[
    (roles["table"] == "snapshot")
    &
    (roles["role"] == "ABLATION_ONLY")
]["column"].tolist()

excluded = roles[
    ~roles["role"].isin(
        [
            "PRIMARY_MODEL_INPUT",
            "ABLATION_ONLY"
        ]
    )
][
    [
        "table",
        "column",
        "role"
    ]
].to_dict(
    orient="records"
)

schema_files = {
    "node_input_features.json":
        node_primary,

    "node_topology_ablation_features.json":
        node_ablation,

    "edge_input_features.json":
        edge_primary,

    "graph_context_features.json":
        snapshot_primary,

    "graph_current_context_ablation_features.json":
        snapshot_ablation,
}

for filename, values in schema_files.items():

    with open(
        OUT / filename,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            values,
            f,
            indent=2
        )


target_definition = {
    "multiclass_target":
        "attackerType",

    "binary_target_definition":
        "1 if attackerType != 0 else 0",

    "known_multiclass_values":
        [0, 1, 2, 4, 8, 16],

    "task_unit":
        "communication edge/message",

    "primary_task":
        "multiclass edge classification",

    "secondary_task":
        "binary benign-vs-malicious edge classification",
}

with open(
    OUT /
    "target_definition.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        target_definition,
        f,
        indent=2
    )


with open(
    OUT /
    "excluded_features.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        excluded,
        f,
        indent=2
    )


print("Node primary:", len(node_primary))
print("Node topology ablation:", len(node_ablation))
print("Edge primary:", len(edge_primary))
print("Previous-graph context:", len(snapshot_primary))
print("Current-graph context ablation:", len(snapshot_ablation))
print("Excluded fields:", len(excluded))


## 8. Experimental feature groups for ablation

These groups will let Paper 1 quantify what is gained by progressively adding temporal and graph information.


In [ ]:
feature_groups = {
    "E1_node_physical_plus_message": {
        "node": [
            x for x in node_primary
            if not x.startswith("prev_")
        ],
        "edge": [
            x for x in edge_primary
            if x not in {
                "prior_pair_message_count",
                "time_since_previous_pair_message",
                "prior_sender_message_count",
                "prior_receiver_message_count",
            }
        ],
        "graph_context": []
    },

    "E2_plus_temporal_history": {
        "node": node_primary,
        "edge": edge_primary,
        "graph_context": []
    },

    "E3_plus_previous_graph_context": {
        "node": node_primary,
        "edge": edge_primary,
        "graph_context": snapshot_primary
    },

    "E4_topology_feature_ablation": {
        "node": node_primary + node_ablation,
        "edge": edge_primary,
        "graph_context": snapshot_primary
    },

    "E5_current_graph_context_ablation": {
        "node": node_primary,
        "edge": edge_primary,
        "graph_context": snapshot_primary + snapshot_ablation
    }
}

with open(
    OUT /
    "ablation_feature_groups.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_groups,
        f,
        indent=2
    )

for k, v in feature_groups.items():
    print(
        k,
        "node=", len(v["node"]),
        "edge=", len(v["edge"]),
        "graph=", len(v["graph_context"])
    )


## 9. Final audit report


In [ ]:
fail_integrity = int(
    (
        training_integrity[
            "integrity_status"
        ] != "PASS"
    ).sum()
)

unclassified = int(
    (
        roles[
            "role"
        ]
        ==
        "UNCLASSIFIED_REVIEW"
    ).sum()
)

potential_leakage_review = int(
    (
        roles[
            "role"
        ]
        ==
        "POTENTIAL_LEAKAGE_REVIEW"
    ).sum()
)

non_numeric_count = len(
    non_numeric_primary
)

target_present = (
    (
        roles[
            "role"
        ]
        ==
        "TARGET"
    ).any()
)

checks = [
    {
        "check":
            "Target identified",

        "status":
            "PASS"
            if target_present
            else "FAIL",

        "detail":
            "attackerType"
    },

    {
        "check":
            "No non-numeric primary features",

        "status":
            "PASS"
            if non_numeric_count == 0
            else "FAIL",

        "detail":
            str(
                non_numeric_count
            )
    },

    {
        "check":
            "Training primary feature integrity",

        "status":
            "PASS"
            if fail_integrity == 0
            else "FAIL",

        "detail":
            f"{fail_integrity} failing feature checks"
    },

    {
        "check":
            "No unclassified fields",

        "status":
            "PASS"
            if unclassified == 0
            else "REVIEW",

        "detail":
            f"{unclassified} unclassified"
    },

    {
        "check":
            "Potential leakage fields conservatively excluded",

        "status":
            "PASS",

        "detail":
            f"{potential_leakage_review} field(s), including A_flag if present"
    },

    {
        "check":
            "Ground-truth validation fields excluded",

        "status":
            "PASS",

        "detail":
            "groundtruth_* fields are never model inputs"
    },

    {
        "check":
            "Label-derived graph statistic excluded",

        "status":
            "PASS",

        "detail":
            "labelled_attack_edge_fraction excluded"
    },

    {
        "check":
            "Primary GNN avoids manually injecting current topology descriptors",

        "status":
            "PASS",

        "detail":
            "current node topology retained for ablation only"
    },
]

report = pd.DataFrame(
    checks
)

if (
    report[
        "status"
    ] == "FAIL"
).any():

    overall = "FAIL"

elif (
    report[
        "status"
    ] == "REVIEW"
).any():

    overall = "REVIEW_REQUIRED"

else:

    overall = "PASS_FREEZE_FEATURE_SCHEMA"


report.loc[
    len(report)
] = {
    "check":
        "FINAL_FEATURE_SCHEMA_STATUS",

    "status":
        overall,

    "detail":
        "Proceed to PyTorch Geometric construction only if PASS"
}

report.to_csv(
    OUT /
    "07_FINAL_FEATURE_SCHEMA_AUDIT.csv",
    index=False
)

display(report)

print("\nFinal feature-schema status:", overall)
print("Outputs:", OUT)


## What to send back

Please upload/paste:

- `02_feature_role_classification.csv`
- `03_features_requiring_manual_review.csv`
- `06_training_primary_feature_integrity_classified.csv`
- `07_FINAL_FEATURE_SCHEMA_AUDIT.csv`
- `node_input_features.json`
- `edge_input_features.json`
- `graph_context_features.json`
- `excluded_features.json`
- `ablation_feature_groups.json`

If the status is `PASS_FREEZE_FEATURE_SCHEMA`, we will freeze the feature contract and then build the GNN-ready graph objects from the optimized train/validation/test split.
